In [35]:
import os

db_path = "out/results_db.sqlite"
out_path = "res"

In [36]:
"""Load the "runs" table from the results SQLite database into a pandas DataFrame."""

import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path(db_path)
def load_runs(db_path: Path = DB_PATH) -> pd.DataFrame:
    # open read-only so we never interfere with a running batch
    with sqlite3.connect(f"file:{db_path}?mode=ro", uri=True) as conn:
        cur = conn.execute("SELECT * FROM runs")
        columns = [d[0] for d in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=columns)


df = load_runs()
print(f"{len(df)} rows x {len(df.columns)} columns")
df.head()
df.to_csv("df.csv")

30 rows x 38 columns


In [37]:
df.columns

Index(['id', 'batch_id', 'job_id', 'timestamp', 'cpw_name', 'framework',
       'model', 'attempt_number', 'status', 'error', 'total_seconds',
       'gen_seconds', 'map_seconds', 'fix_seconds', 'layout_seconds',
       'struct_sim_seconds', 'render_diff_seconds', 'ltl_seconds',
       'cnstr_seconds', 'svg_seconds', 'gold_bpmn', 'ltl_cnstr',
       'cnstr_ontology', 'generated_bpmn', 'mapped_bpmn', 'node_sim',
       'struct_sim', 'ged', 'ltl_score', 'ltl_conf', 'cnstr_score',
       'cnstr_conf', 'cpw_succ_cnstrs', 'cpw_total_cnstrs', 'cpw_score',
       'incl_model_guide', 'incl_knowl', 'optim_prmt'],
      dtype='object')

In [38]:
len(df)

30

In [39]:
df[(df['cpw_name']=='COPD')&(df['incl_knowl']==0)]['struct_sim']

5     0.775056
6     0.797597
7     0.840485
8     0.807061
9     0.850778
15    0.812282
16    0.786372
17    0.833483
18    0.840485
19    0.797597
Name: struct_sim, dtype: float64

In [40]:
def calc_time(df0):
    grouped = df0.groupby('cpw_name')[['total_seconds', 'gen_seconds', 'map_seconds', 'ltl_seconds']].mean().round(0).reset_index()
    grouped['gen_fract'] = (100 / grouped['total_seconds'] * grouped['gen_seconds']).round(0)
    grouped['map_fract'] = (100 / grouped['total_seconds'] * grouped['map_seconds']).round(0)
    grouped['ltl_fract'] = (100 / grouped['total_seconds'] * grouped['ltl_seconds']).round(0)
    grouped.loc[8] = grouped.mean(numeric_only=True).round(1); grouped.loc[8, 'cpw_name'] = "Overall"
    grouped['sum'] = grouped[['gen_seconds', 'map_seconds', 'ltl_seconds']].sum(numeric_only=True, axis=1)
    return grouped

In [41]:
bl = calc_time(df[(df['incl_model_guide']==0)&(df['incl_knowl']==0)])
bl.loc[8, 'cpw_name'] = 'baseline'
mg = calc_time(df[(df['incl_model_guide']==1)&(df['incl_knowl']==0)])
mg.loc[8, 'cpw_name'] = '+ modelling guidelines'
mg_ks = calc_time(df[(df['incl_model_guide']==1)&(df['incl_knowl']==1)])
mg_ks.loc[8, 'cpw_name'] = '+ knowledge constraints'

/var/folders/b6/7128wh613rqcbppftg_0hb2h0000gn/T/ipykernel_79714/2075743284.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Overall' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grouped.loc[8] = grouped.mean(numeric_only=True).round(1); grouped.loc[8, 'cpw_name'] = "Overall"


In [42]:
pd.concat([bl[bl['cpw_name']=='baseline'],mg[mg['cpw_name']=='+ modelling guidelines'],mg_ks[mg_ks['cpw_name']=='+ knowledge constraints']])

,cpw_name,total_seconds,gen_seconds,map_seconds,ltl_seconds,gen_fract,map_fract,ltl_fract,sum
8,baseline,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
8,+ modelling guidelines,51.0,35.5,14.5,0.0,70.0,28.5,0.0,50.0
8,+ knowledge constraints,54.0,41.0,13.0,0.0,76.0,24.0,0.0,54.0


In [43]:
from libs.create_table import create_table

In [44]:
# metrics = ["struct_sim", "ltl_score", "cnstr_score"]
metrics = ["struct_sim"]
table = df.groupby(["incl_model_guide", "incl_knowl", "cpw_name", "optim_prmt"])[metrics].agg(['mean', 'std']).unstack('cpw_name')
create_table(table, incl_optim_prmt=True).to_csv(os.path.join(out_path, "struct_sim.csv"))
create_table(table, incl_optim_prmt=True, merge_std=True).to_csv(os.path.join(out_path, "struct_sim_merged.csv"))

In [45]:
metrics = ["cpw_score"]
table = df.groupby(["incl_model_guide", "incl_knowl", "cpw_name", "optim_prmt"])[metrics].agg(['mean', 'std']).unstack('cpw_name')
create_table(table, incl_optim_prmt=True).to_csv(os.path.join(out_path, "overall_cnstr.csv"))
create_table(table, incl_optim_prmt=True, merge_std=True).to_csv(os.path.join(out_path, "overall_cnstr_merged.csv"))

In [46]:
# from libs.csv_to_latex import csv_to_latex

In [47]:
# print(csv_to_latex(os.path.join(out_path, "struct_sim_merged.csv"),
#                    caption="Structural similarity per CPW",
#                    label="tab:struct-sim", 
#                    text_size='footnotesize', 
#                    bold_max=True))

In [48]:
# print(csv_to_latex(os.path.join(out_path, "ltl_cnstr_merged2.csv"),
#                     caption="Knowledge constraint compliance per CPW",
#                     label="tab:cnstr-sat",
#                     text_size='footnotesize', 
#                     bold_max=True))

In [49]:
df['gen_seconds'].mean()

np.float64(37.65866666666666)

In [50]:
# df = pd.read_csv(os.path.join(out_path, "struct_sim.csv"))
# # df = df[df['Unnamed: 0']=='mean'].drop('Unnamed: 0', axis=1)
# df['bl_mg_diff'] = df['+ modelling guidelines'] - df['baseline']
# df['bl_kc_diff'] = df['+ knowledge constraints'] - df['baseline']
# df['mg_kc_diff'] = df['+ knowledge constraints'] - df['+ modelling guidelines']

In [51]:
# df.mean(numeric_only=True)